## Intro to EDA with Time Series 

> This notebook is a modified version of a notebook from the Berlin Time Series Meetup by Juan Orduz. If you want to dive deeper into time series we can highly reccomend to check out the repository of the meetup on [github](https://github.com/juanitorduz/btsa).

Time series forcasting is another interesting aspect of Data Science. In this notebook we will have a look at some simple plots and methods you can apply when you start with a time series project. We will show on a simple example how to manipulate and plot time series data in python.

#### At the end of this notebook you should:
 * know some helpful plots to start with your time series data
 * know how to decompose a time series and what the single components are

## Setup

All the manipulationsand plots in this notebook can be created with standard libaries such as matplotlib, statsmodels etc. 

In [ ]:
# Main data packages. 
import numpy as np
import pandas as pd

# Data Viz. 
import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.ndimage import gaussian_filter


import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style(
    style='darkgrid', 
    rc={'axes.facecolor': '.9', 'grid.color': '.8'}
)
cmaps_hex = ['#193251','#FF5A36','#1E4485', '#99D04A','#FF5A36', '#DB6668']
sns.set_palette(palette=cmaps_hex)
sns_c = sns.color_palette(palette=cmaps_hex)
%matplotlib inline
from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

## Import Data 

The data for this notebook was downloaded from the [meteoblue website](https://www.meteoblue.com/en/weather/archive/export/basel_switzerland_2661604) and consits of weather data for the city of Basel from 2008 till 2020. 

In [ ]:
# Import data
raw_df = pd.read_csv('data/basel_weather.csv')
raw_df.head()

In [ ]:
raw_df.info()

## Format Data & Feature Engineering

The dataset includes a timestamp. It might be interesting to have a look at additional features like for example `month`, `day`, `day of the year`, and `hour`.

In [ ]:
# Create working copy of dataframe
data_df = raw_df.copy()

# Rename columns in a more pythonic way
data_df = data_df.rename(columns={
    'Basel Temperature [2 m elevation corrected]': 'temperature', 
    'Basel Precipitation Total': 'precipitation', 
    'Basel Wind Speed [10 m]': 'wind_speed', 
    'Basel Wind Direction [10 m]': 'wind_direction'
    }
)

# Convert timestamp to datetime object
# Extract additional features from timestamp column
data_df = data_df.assign(
    timestamp = lambda x: pd.to_datetime(x['timestamp']), 
    date = lambda x: x['timestamp'].dt.date,
    year = lambda x: x['timestamp'].dt.year,
    month = lambda x: x['timestamp'].dt.month,
    day = lambda x: x['timestamp'].dt.day,
    dayofyear = lambda x: x['timestamp'].dt.dayofyear,
    hour = lambda x: x['timestamp'].dt.hour,
)

data_df.head()

## Visualize the Data

Let us start by plotting the temperature data over time. We can either use the hourly development or aggregate the feature by day to get a less dense plot. 

In [ ]:
# Temperature hourly development over time 
fig, ax = plt.subplots()
sns.lineplot(x='timestamp', y='temperature', data=data_df, ax=ax)
ax.set(title='Basel - Temperature (Hourly)', ylabel=r'$^\circ$C');

In [ ]:
# Aggregate temperature by day
daily_data_df = data_df \
    .groupby(['date', 'year', 'month', 'day', 'dayofyear'], as_index=False)\
    .agg({'temperature': np.mean}) \
    .set_index('date')

In [ ]:
# Plot temperature on daily basis 
fig, ax = plt.subplots()
sns.lineplot(x='date', y='temperature', data=daily_data_df.reset_index(), ax=ax)
ax.set(title='Basel - Temperature (Daily)', ylabel=r'$^\circ$C');

Both plots show a clear yearly seasonality but other effects like a trend for example are hard to recognize.

We can focus on the seasonality and create a plot to show the yearly seasonality of temperature and its variation:

In [ ]:
# Plot yearly seasonality
fig, ax = plt.subplots() 

pd.pivot_table(data=daily_data_df[['year', 'dayofyear', 'temperature']], index='dayofyear', columns='year') \
    ['temperature'] \
    .plot(cmap='viridis', alpha=0.5, ax=ax)

ax.legend(title='year', loc='center left', bbox_to_anchor=(1, 0.5))
ax.set(title='Basel - Yearly Temperature (Daily)', ylabel=r'$^\circ$C');

We can also create a *polar plot* that depicts the seasonal change over the year in a circle. This way to visualize the change can be seen in the next plot. 

In [ ]:
# Polar plot for seasonality 
ax = plt.subplot(111, projection='polar')

# Convert and plot data
daily_data_df \
    .assign(day_of_year_cyclic = lambda x: x['dayofyear'].transform(lambda x: 2*np.pi*x/365.5)) \
    .pipe((sns.lineplot, 'data'), 
        x='day_of_year_cyclic', 
        y='temperature', 
        ax=ax
    )

ax.set_ylabel('')
ax.set_title('Basel Temperature (Day of The Year)', va='bottom');

## Smoothing

Usuall the first question of data analysis is "Can we extract general, global pattern here?"
In time series with cyclic seasonal variation, this means that we want to smooth out this seasonality and let the global trend manifest itself.
Most classical, first go-to way for this smoothing is to use [Moving Average](https://en.wikipedia.org/wiki/Moving_average). 

In [ ]:
# Plot moving average of different length (week, month, year)
ma = [7, 30, 356]

smooth_daily_data_df = daily_data_df \
    .reset_index() \
    .assign(date = lambda x: x['date'].transform(pd.to_datetime))

# Smooth and plot
fig, ax = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)

for i, m in enumerate(ma):

    smooth_daily_data_df[f'temp_smooth_ma_{m}'] = smooth_daily_data_df['temperature'].rolling(window=m).mean()

    sns.lineplot(x='date', y='temperature', label='temperature', data=smooth_daily_data_df, alpha=0.5, ax=ax[i])
    sns.lineplot(x='date', y=f'temp_smooth_ma_{m}', label=f'temp_smooth_ma_{m}', data=smooth_daily_data_df, color=sns_c[i + 1], ax=ax[i])
    ax[i].legend(loc='upper left')
    ax[i].set(title='', ylabel=r'$^\circ$C');

plt.suptitle('Basel Temperature (Daily) - Smooth Moving Average', y=1.02);

Using the moving average of 356 units relfects the yearly structure of the data (lowest panel in the plot above).

Another way of smoothing the data is by applying a [Gaussian Filter](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.gaussian_filter.html).

In [ ]:
# Smooth data with gaussion filter
smooth_daily_data_df = smooth_daily_data_df \
    .assign(temp_smooth_gf_30 = lambda x: gaussian_filter(input=x['temperature'], sigma=30)) \
    .assign(temp_smooth_gf_90 = lambda x: gaussian_filter(input=x['temperature'], sigma=90))

# Plot data
fig, ax = plt.subplots()
sns.lineplot(x='date', y='temperature', label='temperature', data=smooth_daily_data_df, alpha=0.5, ax=ax)
sns.lineplot(x='date', y='temp_smooth_gf_30', label='temp_smooth_gf_30', data=smooth_daily_data_df, color=sns_c[1], ax=ax)
sns.lineplot(x='date', y='temp_smooth_gf_90', label='temp_smooth_gf_90', data=smooth_daily_data_df, color=sns_c[2], ax=ax)
ax.legend(loc='lower right')
ax.set(title='Basel Temperature (Daily) -  Smooth Gaussian Filter', ylabel=r'$^\circ$C');

Let us use this smoothing to plot again the seasonality yearly temperature:

In [ ]:
# Plot seasonal yearly temperature 
fig, ax = plt.subplots() 

pd.pivot_table(data=smooth_daily_data_df[['year', 'dayofyear', 'temp_smooth_gf_30']], index='dayofyear', columns='year') \
    ['temp_smooth_gf_30'] \
    .plot(cmap='viridis', alpha=0.5, ax=ax)

ax.legend(title='year', loc='center left', bbox_to_anchor=(1, 0.5))
ax.set(title='Yearly Basel (Smooth - GF) Temperature (Daily)', ylabel=r'$^\circ$C');

## Time Series Decomposition

We can go deeper and decompose time series data into the different ingredients to understand what the sources of variation are we see in the data.
The method `seasonal_decompose` from `statsmodels.tsa.seasonal` can help us to decompose the daily data. This method is based on moving averages and returns the **trend**, **seasonality** and **residuals** of our timeseries.

The stpes inside of this function is:
* we first check for global trends using Moving average
* we de-trend our data by substituding the trend values from the original data
* then we estimate seasonal component by taking averages of each season's value - for example, the effect of January is the average of all de-trended January values in the data

We can define if we want to extract the yearly or monthly trends. Let's have a look at both:

In [ ]:
# We use the parameter `period` = 365 to extract the yearly seasonality. 
seas_decomp_yearly = seasonal_decompose(
    x=daily_data_df['temperature'], 
    model='additive', 
    period=365
)

# Plot data
fig, ax = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True)

seas_decomp_yearly.observed.plot(c=sns_c[0], ax=ax[0])
ax[0].set(title='observed', ylabel=r'$^\circ$C')
seas_decomp_yearly.trend.plot(c=sns_c[1], ax=ax[1])
ax[1].set(title='trend', ylabel=r'$^\circ$C')
seas_decomp_yearly.seasonal.plot(c=sns_c[2], ax=ax[2])
ax[2].set(title='seasonal', ylabel=r'$^\circ$C')
seas_decomp_yearly.resid.plot(c=sns_c[3], ax=ax[3])
ax[3].set(title='residual', ylabel=r'$^\circ$C');

We can now decompose the seasonal component from above. 

In [ ]:
seas_decomp_monthly = seasonal_decompose(
    x=seas_decomp_yearly.seasonal, 
    model='additive', 
    period=52
)

# Plot data
fig, ax = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True)

seas_decomp_monthly.observed.plot(c=sns_c[0], ax=ax[0])
ax[0].set(title='observed', ylabel=r'$^\circ$C')
seas_decomp_monthly.trend.plot(c=sns_c[1], ax=ax[1])
ax[1].set(title='trend', ylabel=r'$^\circ$C')
seas_decomp_monthly.seasonal.plot(c=sns_c[2], ax=ax[2])
ax[2].set(title='seasonal', ylabel=r'$^\circ$C')
seas_decomp_monthly.resid.plot(c=sns_c[3], ax=ax[3])
ax[3].set(title='residual', ylabel=r'$^\circ$C');

### Intuition Behind Time Series Decomposition

The main idea is to model each component separately. Let us see how to do it using [Fourier modes](https://en.wikipedia.org/wiki/Fourier_series).

In [ ]:
# Create linear trend and cyclical variables to model the day of the year. 
smooth_daily_data_df = smooth_daily_data_df.assign(
    index = lambda x: np.linspace(start=0, stop=x.shape[0]-1, num=x.shape[0]),
    day_of_year_cs = lambda x: np.sin(2*np.pi*x['dayofyear']/365.5),
    day_of_year_cc = lambda x: np.cos(2*np.pi*x['dayofyear']/365.5)
)

# Plot data
fig, ax = plt.subplots(2, 1, constrained_layout=True)
sns.lineplot(x='date', y='day_of_year_cs', data=smooth_daily_data_df, color=sns_c[0], ax=ax[0])
ax[0].set(title='day_of_year_cs', ylabel='')
sns.lineplot(x='date', y='day_of_year_cc', data=smooth_daily_data_df, color=sns_c[1], ax=ax[1])
ax[1].set(title='day_of_year_cc', ylabel='')
plt.suptitle('Fourier Mode (Day of the Year)', y=1.05);

Next, we use a linear model to fit each component:

In [ ]:
# Define model
decomp_model = smf.ols(formula = 'temperature ~ index + day_of_year_cs + day_of_year_cc', data=smooth_daily_data_df)

# Train model
decomp_red = decomp_model.fit()
print(decomp_red.summary()) 

Finally, let us extract each component.

In [ ]:
smooth_daily_data_df = smooth_daily_data_df.assign(
    decomp_trend = decomp_red.predict(smooth_daily_data_df.assign(day_of_year_cs = 0.0, day_of_year_cc = 0.0)),
    decomp_seas = decomp_red.predict(smooth_daily_data_df.assign(index = 0.0)) - decomp_red.params['Intercept'],
    decomp_error = lambda x: x['temperature'] - x['decomp_trend'] - x['decomp_seas']
)

fig, ax = plt.subplots(4, 1, figsize=(12, 12), constrained_layout=True)

sns.lineplot(x='date', y='temperature', data=smooth_daily_data_df, color=sns_c[0], ax=ax[0])
ax[0].set(title='observed', ylabel=r'$^\circ$C')

sns.lineplot(x='date', y='decomp_trend', data=smooth_daily_data_df, color=sns_c[1], ax=ax[1])
ax[1].set(title='trend', ylabel=r'$^\circ$C')

sns.lineplot(x='date', y='decomp_seas', data=smooth_daily_data_df, color=sns_c[2], ax=ax[2])
ax[2].set(title='seasonal', ylabel=r'$^\circ$C')

sns.lineplot(x='date', y='decomp_error', data=smooth_daily_data_df, color=sns_c[3], ax=ax[3])
ax[3].set(title='residual', ylabel=r'$^\circ$C');

plt.savefig(f'../images/basel_daily_decomp_fourier.png', dpi=200, bbox_inches='tight');

The results are similar as above.